# 08 — Multimodal Similarity Ranking (RRF)
Computes cosine similarity between a reference molecule (Adagrasib) and
the candidate compounds across three molecular representation spaces,
then integrates per-modality rankings via Reciprocal Rank Fusion (RRF).

**Reference:** Cormack, G. V., Clarke, C. L. A., & Buettcher, S. (2009).
*Reciprocal rank fusion outperforms condorcet and individual rank learning methods.*
SIGIR 2009. https://doi.org/10.1145/1571941.1572114

## Configuration

In [ ]:
# Dataset embedding CSVs 
PATH_FP_DATASET    = "data/fingerprint_embeddings.csv"
PATH_IMG_DATASET   = "data/image_embeddings.csv"
PATH_GRAPH_DATASET = "data/graph_embeddings.csv"

# Reference molecule (Adagrasib) embedding CSVs 
PATH_FP_REF    = "data/adagrasib_fingerprint.csv"
PATH_IMG_REF   = "data/adagrasib_image.csv"
PATH_GRAPH_REF = "data/adagrasib_graph.csv"

# Candidate molecule indices 
PATH_INDICES = "data/candidate_molecules.csv"  # columns: Index, Compound_ID

# Output 
OUTPUT_CSV = "data/molecular_ranking_rrf.csv"

# RRF constant 
K_RRF = 60

print("Configuration loaded.")

## Dependencies

In [ ]:
# pip install numpy pandas scikit-learn

## 1. Imports and helper functions

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


def load_embeddings(path: str, drop_first_col: bool = False) -> np.ndarray:
    """
    Load an embedding CSV (first row = numeric header, skipped automatically).
    If drop_first_col=True, drop the first column (e.g. fingerprint index column).
    """
    df = pd.read_csv(path, header=0)
    if drop_first_col:
        df = df.iloc[:, 1:]
    return df.values.astype(np.float32)


def compute_cosine_similarities(reference: np.ndarray,
                                candidates: np.ndarray) -> np.ndarray:
    """
    Cosine similarity between a reference vector (1, D)
    and a candidate matrix (N, D). Returns 1D array (N,).
    """
    return cosine_similarity(reference.reshape(1, -1), candidates).flatten()


def reciprocal_rank_fusion(sim_dict: dict, k: int = 60) -> np.ndarray:
    """
    For each modality, converts similarities to ranks (rank 1 = most similar)
    and computes the RRF contribution: 1 / (k + rank).
    Returns the sum of contributions across all modalities.
    """
    n = len(next(iter(sim_dict.values())))
    rrf_scores = np.zeros(n, dtype=np.float64)
    for modality, sims in sim_dict.items():
        order = np.argsort(-sims)
        ranks = np.empty_like(order)
        ranks[order] = np.arange(1, n + 1)
        rrf_scores += 1.0 / (k + ranks)
        print(f"  [{modality}] ranks assigned: 1-{ranks.max()}")
    return rrf_scores


print("Functions defined.")

## 2. Load candidate indices

In [ ]:
df_indices        = pd.read_csv(PATH_INDICES)
candidate_indices = df_indices["Index"].tolist()
compound_ids      = df_indices["Compound_ID"].tolist()

print(f"{len(candidate_indices)} candidate molecules loaded.")
display(df_indices)

## 3. Load dataset embeddings

In [ ]:
emb_fp    = load_embeddings(PATH_FP_DATASET)
emb_img   = load_embeddings(PATH_IMG_DATASET)
emb_graph = load_embeddings(PATH_GRAPH_DATASET)

print(f"Fingerprint : {emb_fp.shape}   (expected: N x 1024)")
print(f"Image       : {emb_img.shape}   (expected: N x 12544)")
print(f"Graph       : {emb_graph.shape} (expected: N x 128)")

## 4. Load reference molecule embeddings

In [ ]:
ref_fp    = load_embeddings(PATH_FP_REF)
ref_img   = load_embeddings(PATH_IMG_REF)
ref_graph = load_embeddings(PATH_GRAPH_REF)

print(f"Reference fingerprint : {ref_fp.shape}   (expected: 1 x 1024)")
print(f"Reference image       : {ref_img.shape}   (expected: 1 x 12544)")
print(f"Reference graph       : {ref_graph.shape} (expected: 1 x 128)")

## 5. Extract candidate embeddings

In [ ]:
cand_fp    = emb_fp[candidate_indices]
cand_img   = emb_img[candidate_indices]
cand_graph = emb_graph[candidate_indices]

print(f"Candidate fingerprint : {cand_fp.shape}")
print(f"Candidate image       : {cand_img.shape}")
print(f"Candidate graph       : {cand_graph.shape}")

## 6. Compute cosine similarities

In [ ]:
sims_fp    = compute_cosine_similarities(ref_fp,    cand_fp)
sims_img   = compute_cosine_similarities(ref_img,   cand_img)
sims_graph = compute_cosine_similarities(ref_graph, cand_graph)

display(pd.DataFrame({
    "Compound_ID"    : compound_ids,
    "sim_fingerprint": sims_fp.round(6),
    "sim_image"      : sims_img.round(6),
    "sim_graph"      : sims_graph.round(6),
}))

## 7. Reciprocal Rank Fusion

In [ ]:
print(f"Applying RRF (k={K_RRF})...")
rrf_scores = reciprocal_rank_fusion(
    {"fingerprint": sims_fp, "image": sims_img, "graph": sims_graph},
    k=K_RRF
)
print("RRF complete.")

## 8. Final ranking

In [ ]:
results = pd.DataFrame({
    "Compound_ID"     : compound_ids,
    "Dataset_Index"   : candidate_indices,
    "sim_fingerprint" : sims_fp.round(6),
    "sim_image"       : sims_img.round(6),
    "sim_graph"       : sims_graph.round(6),
    "rrf_score"       : rrf_scores.round(8),
})

for col in ["sim_fingerprint", "sim_image", "sim_graph"]:
    results[f"rank_{col.split('_')[1]}"] = results[col].rank(ascending=False).astype(int)

results["rank_rrf"] = results["rrf_score"].rank(ascending=False).astype(int)
results["rank_std"] = results[["rank_fingerprint", "rank_image", "rank_graph"]].std(axis=1).round(3)
results = results.sort_values("rank_rrf").reset_index(drop=True)

cols_show = [
    "Compound_ID",
    "sim_fingerprint", "rank_fingerprint",
    "sim_image",       "rank_image",
    "sim_graph",       "rank_graph",
    "rrf_score"
]
print("FINAL RANKING (RRF):")
display(results[cols_show])

## 9. Save results

In [ ]:
results.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")